In [1]:
#Inspired by https://avandekleut.github.io/vae/
import torch;
import torch.nn as nn
import torch.utils
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import wandb
wandb.login()

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

wandb: Currently logged in as: last24ag (last24ag-copenhagen-business-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [3]:
class FF(nn.Module):
    def __init__(self,dim1,dim2,dim3):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(in_features=dim1, out_features=dim2),
            nn.ReLU(),
            nn.Linear(in_features=dim2, out_features=dim3)
        )

    def forward(self, input):
        return self.main(input)
tmp = FF(28*28,512,2)
print(tmp)
print(tmp(torch.rand(10,1,28*28)).shape)

FF(
  (main): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=2, bias=True)
  )
)
torch.Size([10, 1, 2])


In [4]:
class Autoencoder(nn.Module):
    def __init__(self, dim1, dim2, dim3):
        super().__init__()
        self.encoder = FF(dim1, dim2, dim3)
        self.decoder = nn.Sequential(
            FF(dim3, dim2, dim1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
tmp = Autoencoder(28*28,512,2)
print(tmp)
print(tmp(torch.rand(10,1,28*28)).shape)

Autoencoder(
  (encoder): FF(
    (main): Sequential(
      (0): Linear(in_features=784, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=2, bias=True)
    )
  )
  (decoder): Sequential(
    (0): FF(
      (main): Sequential(
        (0): Linear(in_features=2, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=784, bias=True)
      )
    )
    (1): Sigmoid()
  )
)
torch.Size([10, 1, 784])


In [5]:
def train(data_loader, model, optimizer, loss_function, epochs=20):
    model.to(device)
    losses = []
    global_step = 0

    for epoch in range(epochs):
        for i, (x, y) in enumerate(data_loader):
            x = x.to(device)

            optimizer.zero_grad()
            x_hat = model(x)
            loss = loss_function(x, x_hat)

            losses.append(loss.item())
            loss.backward()
            optimizer.step()

            # log to W&B
            wandb.log({"train_loss": loss.item(), "epoch": epoch}, step=global_step)
            global_step += 1

            if i % 100 == 0:
                print(f"{epoch=}/{i}: {loss}")
    return model, losses

In [6]:
def plot_latent(data_loader, encoder, dim1=0, dim2=1, num_batches=100):
    for i, (x, y) in enumerate(data_loader):
        z = encoder(x.to(device))
        z = z.to('cpu').detach().numpy()
        plt.scatter(z[:, 0, dim1], z[:, 0, dim2], c=y, alpha=0.5)
        if i > num_batches:
            plt.colorbar()
            break

In [7]:
def plot_reconstructed(decoder, w, h, r0=(-10, 10), r1=(-10, 10), n=12):
    img = np.zeros((n*w, n*h))
    for i, y in enumerate(np.linspace(*r1, n)):
        for j, x in enumerate(np.linspace(*r0, n)):
            z = torch.Tensor([[x, y]]).view(1,1,2).to(device)
            x_hat = decoder(z)
            x_hat = x_hat.reshape(w, h).to('cpu').detach().numpy()
            img[(n-1-i)*w:(n-1-i+1)*w, j*w:(j+1)*w] = x_hat
    plt.imshow(img, extent=[*r0, *r1])

In [8]:
# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.flatten(x,start_dim=-2))
])

data = datasets.MNIST('./data',transform=transform,download=True)

n, w, h = data.data.shape

data_loader = torch.utils.data.DataLoader(data,batch_size=128,shuffle=True)

LATENT_DIM = 2
HIDDEN_DIM = 512
EPOCHS = 20
BATCH_SIZE = 128

run = wandb.init(
    project="mnist-autoencoder",
    config=dict(
        latent_dim=LATENT_DIM,
        hidden_dim=HIDDEN_DIM,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
    )
)


model = Autoencoder(w * h, HIDDEN_DIM, LATENT_DIM)
optimizer = torch.optim.Adam(model.parameters())
loss_function = torch.nn.MSELoss()

(autoencoder, losses) = train(data_loader, model, optimizer, loss_function, EPOCHS)

epoch=0/0: 0.230881005525589
epoch=0/100: 0.05784895643591881
epoch=0/200: 0.05303632467985153
epoch=0/300: 0.04772034287452698
epoch=0/400: 0.04938548058271408
epoch=1/0: 0.04810505360364914
epoch=1/100: 0.046630095690488815
epoch=1/200: 0.0461246520280838
epoch=1/300: 0.04531022161245346
epoch=1/400: 0.046350643038749695
epoch=2/0: 0.044378407299518585
epoch=2/100: 0.045265719294548035
epoch=2/200: 0.04644595831632614
epoch=2/300: 0.04214209318161011
epoch=2/400: 0.04306016489863396
epoch=3/0: 0.04238414391875267
epoch=3/100: 0.04378620535135269
epoch=3/200: 0.04330664873123169
epoch=3/300: 0.04482369124889374
epoch=3/400: 0.04134116321802139
epoch=4/0: 0.04481283575296402
epoch=4/100: 0.04573625326156616
epoch=4/200: 0.04235076531767845
epoch=4/300: 0.042664118111133575
epoch=4/400: 0.04494663327932358
epoch=5/0: 0.045787252485752106
epoch=5/100: 0.043642595410346985
epoch=5/200: 0.04148922115564346
epoch=5/300: 0.04136868566274643
epoch=5/400: 0.043147530406713486
epoch=6/0: 0.0443

In [9]:
# Defining the Plot Style
plt.xlabel('Iterations')
plt.ylabel('Loss')

# Plotting the losses
plt.plot(losses)
#plt.savefig("loss_curve.png")
wandb.log({"loss_curve": wandb.Image(plt.gcf())})
plt.close()

In [10]:
# Plot latent space
plot_latent(data_loader, autoencoder.encoder)
#plt.savefig('latent.pdf')
wandb.log({"latent_space": wandb.Image(plt.gcf())})
plt.close()

In [11]:
# Let's collect all images and labels from 'data' (which is likely a TensorDataset)
X = []
y = []

for xi, yi in data:
    X.append(xi.numpy())
    y.append(yi.numpy() if hasattr(yi, 'numpy') else yi)

X = np.stack(X, axis=0)
y = np.array(y)

# X may be (N, 784) or (N, 1, 28, 28) depending on the earlier transform; ensure the shape
if X.ndim > 2:
    X = X.reshape(X.shape[0], -1)

print("X shape for PCA:", X.shape)

LATENT_DIM_PCA = 2
pca = PCA(n_components=LATENT_DIM_PCA)
Z = pca.fit_transform(X) # latent representation (N, 2)
X_hat = pca.inverse_transform(Z) # reconstruction

# Reconstruction error (MSE over all pixels and samples)
reconstruction_mse = float(np.mean((X - X_hat) ** 2))
explained_variance_ratio = float(np.sum(pca.explained_variance_ratio_))

print("PCA reconstruction MSE:", reconstruction_mse)
print("PCA explained variance (2D):", explained_variance_ratio)

wandb.log({
    "pca_reconstruction_mse": reconstruction_mse,
    "pca_explained_variance_ratio_2d": explained_variance_ratio,
})

X shape for PCA: (60000, 784)
PCA reconstruction MSE: 0.055952705442905426
PCA explained variance (2D): 0.1680038869380951


In [12]:
def plot_pca_latent(Z, labels, num_points=10000):
    n = len(Z)
    num_points = min(num_points, n)
    idx = np.random.choice(n, size=num_points, replace=False)


    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z[idx, 0], Z[idx, 1], c=labels[idx], s=5, alpha=0.6)
    plt.colorbar(sc, ticks=range(10), label="Digit")
    plt.xlabel("PC 1")
    plt.ylabel("PC 2")
    plt.title("MNIST in 2D PCA latent space")
    plt.tight_layout()

plot_pca_latent(Z, y)
plt.savefig("latent_pca.png", dpi=150)
wandb.log({"latent_space_pca": wandb.Image(plt.gcf())})
plt.close()

In [ ]:
# Plot reconstruction error versus number of latent nodes

# Ensure wandb is initialized before logging
if wandb.run is None:
    wandb.init(project="mnist-autoencoder", reinit=True)

# Create train/test split
train_size = int(0.8 * len(data))
test_size = len(data) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(data, [train_size, test_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# Collect test data for PCA evaluation
X_test = []
for xi, yi in test_dataset:
    x_np = xi.numpy()
    if x_np.ndim > 1:
        x_np = x_np.reshape(-1)
    X_test.append(x_np)
X_test = np.stack(X_test, axis=0)

latent_dims = list(range(1, 21))
reconstruction_errors = []
pca_reconstruction_errors = []

# Compute PCA reconstruction errors for each latent dimension
for d in latent_dims:
    pca = PCA(n_components=d)
    Z_pca = pca.fit_transform(X_test)
    X_hat_pca = pca.inverse_transform(Z_pca)
    mse_pca = float(np.mean((X_test - X_hat_pca) ** 2))
    pca_reconstruction_errors.append(mse_pca)

# Compute Autoencoder reconstruction errors for each latent dimension
for d in latent_dims:
    # Autoencoder(input_dim, hidden_dim, latent_dim)
    model = Autoencoder(784, 256, d).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.MSELoss()
    num_epochs = 5

    for epoch in range(num_epochs):
        model.train()
        for x, _ in train_loader:
            x = x.to(device)
            if x.dim() == 3:
                x = x.squeeze(1)
            optimizer.zero_grad()
            x_hat = model(x)
            loss = loss_fn(x_hat, x)
            loss.backward()
            optimizer.step()
    
    # Evaluate on the test set
    model.eval()
    mse = 0.0
    n = 0
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            if x.dim() == 3:
                x = x.squeeze(1)
            x_hat = model(x)
            mse += ((x_hat - x) ** 2).sum().item()
            n += x.numel()
    mse /= n
    reconstruction_errors.append(mse)

# Plot both autoencoder and PCA reconstruction errors
plt.figure(figsize=(7, 4))
plt.plot(latent_dims, reconstruction_errors, marker="o", label="Autoencoder", linewidth=2)
plt.plot(latent_dims, pca_reconstruction_errors, marker="s", label="PCA", linewidth=2)
plt.xlabel("Latent dimension (dim3)")
plt.ylabel("Reconstruction MSE")
plt.title("Reconstruction error vs. number of latent nodes")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("reconstruction_error_vs_latent_dim.png", dpi=150)
#plt.show()  
# Ensure wandb logging only happens if run is initialized
if wandb.run is not None:
    wandb.log({
        "recon_error_vs_latent_dim": wandb.Image(plt.gcf()),
        "reconstruction_errors": reconstruction_errors,
        "pca_reconstruction_errors": pca_reconstruction_errors
    })
plt.close()
#close wandb
wandb.close()

## **Answers to Task 2.1** 

### 2.1.1 Autoencoder Program Walkthrough

**Overview:** The notebook starts by importing PyTorch building blocks (modules, data utilities, optimizers), torchvision for MNIST, and numpy/matplotlib for analysis. It immediately detects whether CUDA is available and stores the chosen `device`, keeping tensor placement consistent across CPU/GPU during training and visualization.

**Functions:** 

The reusable `FF` module is a two-layer MLP with a ReLU in the middle; it is the core unit for both encoder and decoder. 

`Autoencoder` chains two of these blocks: the encoder compresses each 784-dimensional flattened image into a 2-D latent code, and the decoder expands that code back to 784 features, finishing with a Sigmoid so reconstructed pixels stay in range. 

The `train` routine drives optimization by iterating over DataLoader batches, moving data to `device`, running forward passes, measuring reconstruction loss, backpropagating, and stepping Adam, with periodic loss prints every 100 mini-batches. 

At last, `plot_latent` runs the encoder on batches and scatters the resulting 2-D codes colored by digit labels, while `plot_reconstructed` sweeps a grid of latent points through the decoder to show what images different latent regions produce.


### 2.1.2 The transform function

The `transform` first converts $\texttt{MNIST PIL}$ images to float tensors scaled to $[0,1]$ via \lstinline{ToTensor}, then flattens each 28×28 grid into a 784-length vector using a \lstinline{Lambda} \lstinline{flatten} so it fits the linear layers. \texttt{MNIST} is downloaded to \texttt{./data} if missing and served by a shuffled \lstinline{DataLoader} with batch size 128, ensuring randomized mini-batches throughout training. The flatten step is essential here because the model is fully connected rather than convolutional.

**Model Definition:** In the encoder, a ReLU after the first linear layer captures non-linear structure, while the final latent layer is left linear so the 2-D space remains unconstrained and easy to visualize. The decoder mirrors this pattern: a ReLU after the latent-to-hidden projection adds expressiveness, and a final Sigmoid bounds outputs to `[0,1]`, aligning with normalized pixel intensities and the reconstruction objective.

**Training & Loss:** Training uses Adam over all parameters. Reconstruction quality is measured with mean squared error between the input and decoded image. This choice matches the Gaussian-with-shared-variance view of pixel noise; paired with inputs scaled to `[0,1]` and a Sigmoid output layer, MSE provides a simple, stable signal for learning faithful reconstructions.

**Alternative Likelihood (Beta):** To better respect data constrained to `[0,1]`, the decoder could emit Beta distribution parameters per pixel instead of a single mean. Practically, it would output two positive numbers (or a mean plus concentration mapped through `softplus` with a small epsilon) and use the Beta negative log-likelihood as the loss. Inputs are typically clamped slightly away from exact 0 or 1 to avoid log singularities. This keeps outputs in `(0,1)` while allowing pixel-wise variance to adapt, offering a more flexible likelihood than the fixed-variance Gaussian implied by MSE.